# BigQuery Studio – Finance demo notebook

This notebook loads data from BigQuery and GCS, runs finance-oriented analyses, and displays graphs.

**Data sources:**
- BigQuery dataset: `bq-features-489714.bq_studio_demo`
- GCS bucket: `gs://bq-studio-demo-bq-features-489714`


## 1. Connect to BigQuery and load tables as DataFrames

In [ ]:
# Run in Vertex AI Workbench or Colab Enterprise with google-cloud-bigquery and pandas
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='bq-features-489714')
dataset = 'bq_studio_demo'

def bq_to_df(table: str):
    return client.query(f"SELECT * FROM `bq-features-489714.bq_studio_demo`.{table}").to_dataframe()

prices = bq_to_df('daily_prices')
holdings = bq_to_df('portfolio_holdings')
pnl = bq_to_df('pnl_daily')
print(prices.head())
print(pnl.head())


## 2. Time series: close price and simple returns

In [ ]:
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values(['symbol', 'date'])
prices['return'] = prices.groupby('symbol')['close'].pct_change()

# Plot close prices (sample)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for sym in prices['symbol'].unique()[:3]:
    d = prices[prices['symbol'] == sym]
    ax.plot(d['date'], d['close'], label=sym)
ax.set_title('Close prices (sample symbols)')
ax.legend()
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()


## 3. PnL over time (bar chart)

In [ ]:
pnl['date'] = pd.to_datetime(pnl['date'])
pnl['pnl_num'] = pd.to_numeric(pnl['pnl'], errors='coerce')
agg = pnl.groupby('date')['pnl_num'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(agg['date'], agg['pnl_num'], color='steelblue', edgecolor='navy')
ax.set_title('Daily PnL (aggregate)')
ax.set_xlabel('Date')
ax.set_ylabel('PnL')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Using Gemini in BigQuery Studio

In **BigQuery Studio** you can use **Gemini** to:
- Generate SQL from natural language (e.g. "total portfolio value by date")
- Explain existing queries
- Fix errors and suggest optimizations

See `docs/gemini-in-bigquery-studio.md` for step-by-step instructions.
